In [1]:
# !wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('course_lead_scoring_2026.csv')


df.columns.to_list()



['lead_source',
 'industry',
 'employment_status',
 'location',
 'annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score',
 'converted']

In [3]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [4]:
df.interaction_count.nunique()

14

In [5]:
categorical_features = list(df.columns[df.dtypes == 'str'])
categorical_features

['lead_source', 'industry', 'employment_status', 'location']

In [6]:
numerical_features = list(df.columns[(df.dtypes == "int64")| (df.dtypes == "float64")])
numerical_features.remove('converted')
numerical_features

['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score']

In [7]:
df[categorical_features] = df[categorical_features].fillna("NA")

In [8]:
df[categorical_features]

,lead_source,industry,employment_status,location
0,organic_search,technology,employed,europe
1,social_media,technology,employed,south_america
2,referral,retail,employed,europe
3,paid_ads,manufacturing,student,NA
4,referral,manufacturing,student,north_america
...,...,...,...,...
4995,organic_search,education,employed,asia
4996,NA,technology,employed,north_america
4997,organic_search,finance,employed,asia
4998,referral,technology,employed,africa


In [9]:
df[numerical_features] = df[numerical_features].fillna(0.0)

In [10]:
df.isna().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [11]:
df.industry.mode()

0    technology
Name: industry, dtype: str

In [12]:
numerical_features

['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score']

In [13]:
df[numerical_features].corr(method='pearson')

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [14]:
df[numerical_features].corrwith(df.lead_score)

annual_income               0.229496
number_of_courses_viewed    0.757204
interaction_count           0.915746
lead_score                  1.000000
dtype: float64

In [15]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

y_train = df_train.converted.values
y_train
y_valid = df_val.converted.values
y_valid
y_test = df_test.converted.values
y_test

y_train.shape, y_valid.shape, y_test.shape



((3000,), (1000,), (1000,))

In [16]:
del df_train["converted"]
del df_val["converted"]
del df_test["converted"]

In [17]:
from sklearn.metrics import mutual_info_score

for cat in categorical_features:
    print(cat, mutual_info_score(df.converted, df[cat]))

lead_source 0.033063899391404215
industry 0.0022800937730549087
employment_status 0.022797267862555814
location 0.0010636898089210062


In [18]:
from sklearn.feature_extraction import DictVectorizer

features = categorical_features + numerical_features

print(features)

train_dicts = df_train[features].to_dict(orient="records")
valid_dicts = df_val[features].to_dict(orient="records")



print("One customer as a dictionary:")
print(type(train_dicts))
print(train_dicts[0])

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)  # learn the categories + transform
X_valid = dv.transform(valid_dicts)      # only transform (same columns!)

X_train.shape

['lead_source', 'industry', 'employment_status', 'location', 'annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
One customer as a dictionary:
<class 'list'>
{'lead_source': 'paid_ads', 'industry': 'finance', 'employment_status': 'employed', 'location': 'asia', 'annual_income': 65980.0, 'number_of_courses_viewed': 2, 'interaction_count': 3, 'lead_score': 0.37}


(3000, 28)

In [19]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

original_accuracy = model.score(X_valid, y_valid)
original_accuracy

0.645

In [20]:
dv.get_feature_names_out()

array(['annual_income', 'employment_status=NA',
       'employment_status=employed', 'employment_status=self_employed',
       'employment_status=student', 'employment_status=unemployed',
       'industry=NA', 'industry=education', 'industry=finance',
       'industry=healthcare', 'industry=manufacturing', 'industry=retail',
       'industry=technology', 'interaction_count', 'lead_score',
       'lead_source=NA', 'lead_source=events',
       'lead_source=organic_search', 'lead_source=paid_ads',
       'lead_source=referral', 'lead_source=social_media', 'location=NA',
       'location=africa', 'location=asia', 'location=europe',
       'location=north_america', 'location=south_america',
       'number_of_courses_viewed'], dtype=object)

In [21]:
number_of_features = len(dv.get_feature_names_out())
number_of_features

28

In [22]:
X_train.shape

type(X_train)

X_new = np.delete(X_train, 0, 1)
X_new.shape

(3000, 27)

In [26]:
from copy import copy
diff_accuracies = []

for feat in features:
    features_new = copy(features)
    features_new.remove(feat)
    train_dicts = df_train[features_new].to_dict(orient="records")
    valid_dicts = df_val[features_new].to_dict(orient="records")

    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(train_dicts)  # learn the categories + transform
    X_valid = dv.transform(valid_dicts)      # only transform (same columns!)
    model.fit(X_train, y_train)
    accuracy = model.score(X_valid, y_valid)
    diff_accuracies.append((feat, original_accuracy - accuracy))


diff_accuracies

[('lead_source', 0.0030000000000000027),
 ('industry', 0.0),
 ('employment_status', 0.0),
 ('location', 0.0),
 ('annual_income', -0.07899999999999996),
 ('number_of_courses_viewed', 0.0020000000000000018),
 ('interaction_count', 0.04400000000000004),
 ('lead_score', 0.0010000000000000009)]

In [29]:
# Regularized Logistic Regression

accuracies = []

for c in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    accuracy = model.score(X_valid, y_valid)
    accuracies.append((c, accuracy))

print(accuracies)


[(1e-06, 0.598), (1e-05, 0.598), (0.0001, 0.613), (0.001, 0.644)]
